#### LSTM 자연어 모델 문제

1. data 폴더 안에 ratings_train.txt 파일 로드
2. 텍스트 정규화 함수 이용하여 document 컬럼의 텍스트들을 정규화
    - 특수문자 제거, 2칸 이상의 공백을 1칸의 공백으로 대체, 문자 좌우의 공백 제거
3. 공백 텍스트 제거
4. 결측치 제거
5. 중복 데이터 제거
6. 상위 5000개의 데이터 추출
7. komoran을 이용해서 데이터 토큰화
    - 품사 NNP NNG VV VA MAG SL
8. 단어 사전 생성 (min_count = 2)
9. Dataset 생성(인코딩), collate_fn 생성하여 패딩 토큰 채워준다.
10. DataLoader 생성
11. 8:2의 비율로 train, val 데이터셋으로 나눠준다.
11. LSTM 학습 모델 생성
    - Embedding() → LSTM() → Linear()
    - LSTM에서는 마지막 히든층을 이용하여 선형 모델에 대입
12. epoch의 횟수는 50회로 모델 검증

In [184]:
import pandas as pd
import re
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from collections import Counter

from konlpy.tag import Komoran
from tqdm import tqdm

**1**

In [185]:
df = pd.read_csv('../data/ratings_train.txt', sep = '\t')

**4**

In [186]:
df.dropna(inplace = True)

**2**

In [187]:
def normalize(text):
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [188]:
df['document'] = df['document'].map(normalize)

**3**

In [189]:
df = df.loc[
    ~(df['document'] == ''),
]

**5**

In [190]:
df.drop_duplicates('document', inplace = True)

In [191]:
df.reset_index(drop=True, inplace = True)

In [192]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 144733 entries, 0 to 144732
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        144733 non-null  int64
 1   document  144733 non-null  str  
 2   label     144733 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.3 MB


**6**

In [193]:
df2 = df[:5000]

**7**

In [194]:
komoran = Komoran()
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']

def tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            tokens.append(word)
    return tokens

In [195]:
tokenized_sentence = [ tokenize(text) for text in df2['document'] ]

**8**

In [196]:
vocab = {
    '<PAD>': 0,
    '<UNK>': 1
}

# tokenized_sentence에서 모든 토큰을 하나의 리스트로 생성
all_tokens = [ token for tokens in tokenized_sentence for token in tokens ]

# token들의 빈도수를 확인 → min_count로 제한
token_counts = Counter(all_tokens)

In [197]:
for token, count in token_counts.items():
    if count >= 2:
        vocab[token] = len(vocab)

**9**

In [198]:
class LSTMDataset(Dataset):
    def __init__( self, tokenized_texts, labels, vocab ):
        self.labels = labels.values
        self.data = [
            [
                vocab.get(token, vocab['<UNK>']) for token in tokens
            ]
            for tokens in tokenized_texts
        ]
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype = torch.long), \
            torch.tensor(self.labels[idx], dtype = torch.long)

In [199]:
def collate_fn(batch):
    text_list = [item[0] for item in batch]
    label_list = [item[1] for item in batch]

    padded_texts = pad_sequence(text_list, batch_first = True, padding_value = vocab['<PAD>'])
    labels = torch.tensor(label_list, dtype = torch.long)

    return padded_texts, labels

**10, 11**

In [200]:
# Dataset 생성
dataset = LSTMDataset(tokenized_sentence, df2[['label']], vocab)

# train의 길이와 test의 길이를 설정
train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, test_size])

In [201]:
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True, collate_fn = collate_fn)
val_loader = DataLoader(val_dataset, batch_size = 64, shuffle = True, collate_fn = collate_fn)

**12**

In [202]:
class LSTMCLF(nn.Module):

    def __init__(self, vocab_size, emb_dim, hidden_size, num_classes, dropout = 0.5, head_type = 'last'):
        super().__init__()

        self.head_type = head_type

        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx = vocab['<PAD>'])
        
        # 자비에르 초기화
        torch.nn.init.xavier_uniform_(self.emb.weight)
        
        self.lstm = nn.LSTM(emb_dim, hidden_size, batch_first = True)
        # 과적합 방지용 dropout
        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, num_classes)
    

    def forward(self, x):

        embedding = self.emb(x)
        lstm_out, (hidden, cell) = self.lstm(embedding)

        if self.head_type == 'last':
            last_hidden = hidden.squeeze(0)
        elif self.head_type == 'mean':
            # 모든 층의 값들의 평균을 구한다.
            # lstm_out → [batch_size, seq_len, hidden_size]
            last_hidden = torch.mean(lstm_out, dim = 1)     # [batch_size, hidden_size]
        elif self.head_type == 'max':
            last_hidden, _ = torch.max(lstm_out, dim = 1)

        dropout_hidden = self.dropout(last_hidden)

        return self.fc(dropout_hidden)

**13**

In [203]:
# 모델 생성
model = LSTMCLF(len(vocab), emb_dim = 64, hidden_size = 128, num_classes = 2, head_type = 'max')

# 손실 함수
criterion = nn.CrossEntropyLoss()

# 옵티마이저
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [204]:
epochs = 50

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct_train = 0
    total_train = 0

    for inputs, labels in tqdm(train_loader, desc = f'Epoch {epoch+1} / {epochs} Train'):
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
    
        train_loss += loss.item()
        pred = torch.argmax(output, dim = 1)
        correct_train += (pred == labels).sum().item()
        total_train += labels.size(0)
    
    train_acc = (correct_train / total_train) * 100
    avg_train_loss = train_loss / len(train_loader)


    # 검증 구간
    model.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            output = model(inputs)
            loss = criterion(output, labels)
            val_loss += loss.item()
            pred = torch.argmax(output, dim = 1)
            correct_val += (pred == labels).sum().item()
            total_val += labels.size(0)
    
    val_acc = (correct_val / total_val) * 100
    avg_val_loss = val_loss / len(val_loader)

    if (epoch+1) % 10 == 0:
        print(f'RNN epoch - Train Loss: {round(avg_train_loss, 4)} / Train Acc : {train_acc}')
        print(f'RNN epoch - Val Loss: {round(avg_val_loss, 4)} / Val Acc : {val_acc}')

Epoch 10 / 50 Train: 100%|██████████| 63/63 [00:01<00:00, 35.58it/s]


RNN epoch - Train Loss: 0.1868 / Train Acc : 93.95
RNN epoch - Val Loss: 0.8087 / Val Acc : 72.89999999999999


Epoch 20 / 50 Train: 100%|██████████| 63/63 [00:01<00:00, 32.63it/s]


RNN epoch - Train Loss: 0.1333 / Train Acc : 95.85000000000001
RNN epoch - Val Loss: 0.9135 / Val Acc : 70.89999999999999


Epoch 30 / 50 Train: 100%|██████████| 63/63 [00:01<00:00, 42.66it/s]


RNN epoch - Train Loss: 0.1184 / Train Acc : 95.675
RNN epoch - Val Loss: 1.2132 / Val Acc : 71.3


Epoch 40 / 50 Train: 100%|██████████| 63/63 [00:01<00:00, 35.96it/s]


RNN epoch - Train Loss: 0.0961 / Train Acc : 95.775
RNN epoch - Val Loss: 1.7962 / Val Acc : 71.2


Epoch 50 / 50 Train: 100%|██████████| 63/63 [00:01<00:00, 42.15it/s]


RNN epoch - Train Loss: 0.0678 / Train Acc : 96.75
RNN epoch - Val Loss: 1.7954 / Val Acc : 71.2
